# Phase 3b: NFCorpus Medical Reranking

AutoDL GPU — Medical information retrieval passage reranking experiments (cross-domain validation)

| # | Experiment | Method | Dataset | Model |
|---|------------|--------|---------|-------|
| E21 | LoRA NFCorpus Qwen    | LoRA (r=16) | NFCorpus Rerank | Qwen2.5-1.5B |
| E22 | LoRA NFCorpus Llama   | LoRA (r=16) | NFCorpus Rerank | Llama-3.2-1B |
| E23 | Full FT NFCorpus Qwen | Full FT     | NFCorpus Rerank | Qwen2.5-1.5B |
| E24 | Full FT NFCorpus Llama| Full FT     | NFCorpus Rerank | Llama-3.2-1B |
| E25 | Random NFCorpus Qwen  | LoRA (shuffled labels) | NFCorpus Rerank | Qwen2.5-1.5B |
| E26 | Random NFCorpus Llama | LoRA (shuffled labels) | NFCorpus Rerank | Llama-3.2-1B |

## 0. Environment Setup

In [ ]:
import os, sys
os.chdir('/root/MLP')
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'offline'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.makedirs('/root/autodl-tmp/hf_cache', exist_ok=True)
os.makedirs('/root/autodl-tmp/outputs', exist_ok=True)
os.makedirs('/root/autodl-tmp/logs', exist_ok=True)
os.makedirs('/root/autodl-tmp/data', exist_ok=True)

# Symlink to persistent storage
for target, link in [('/root/autodl-tmp/outputs', '/root/MLP/outputs'),
                      ('/root/autodl-tmp/logs',    '/root/MLP/logs'),
                      ('/root/autodl-tmp/data',    '/root/MLP/data')]:
    if not os.path.islink(link):
        if os.path.isdir(link):
            print(f'  ⚠ {link} is a real dir, skipping')
        else:
            os.symlink(target, link)
            print(f'  ✓ created: {link} -> {target}')
    else:
        print(f'  ✓ symlink exists: {link} -> {os.readlink(link)}')

!pwd && ls
print(f'Python: {sys.executable}')
print(f'HF_HOME: {os.environ["HF_HOME"]}')
print(f'HF_ENDPOINT: {os.environ["HF_ENDPOINT"]}')
print(f'WANDB_MODE: {os.environ["WANDB_MODE"]}')

  ✓ symlink exists: /root/MLP/outputs -> /root/autodl-tmp/outputs
  ⚠ /root/MLP/logs is a real dir, skipping
  ✓ symlink exists: /root/MLP/data -> /root/autodl-tmp/data
/root/MLP
README.md		 autodl_run_phase3_nfcorpus.ipynb  data    outputs  src
autodl_run.ipynb	 autodl_run_phase3_pubmed.ipynb    logs    results
autodl_run_phase2.ipynb  configs			   models  scripts
Python: /root/miniconda3/bin/python
HF_HOME: /root/autodl-tmp/hf_cache
HF_ENDPOINT: https://hf-mirror.com
WANDB_MODE: offline


In [ ]:
import sys
!{sys.executable} -m pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub datasets
print('\n✅ Installation complete! If this is the first install, restart the kernel: Kernel → Restart Kernel')

Looking in indexes: http://mirrors.aliyun.com/pypi/simple

✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel


In [3]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA H800 PCIe


In [ ]:
# HuggingFace login (required for Llama)
from huggingface_hub import login
login()
print('HuggingFace login successful')

HuggingFace 登录成功


## 1. Data Download & Formatting

Download `BeIR/nfcorpus` (medical information retrieval) from HuggingFace and build 5-candidate passage reranking task data.

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/data/nfcorpus/nfcorpus_formatting.py --output_dir data/nfcorpus
print('Data formatting complete')

In [ ]:
# Verify data files
import os, json
files = [
    'data/nfcorpus/train_sft.jsonl',
    'data/nfcorpus/val_sft.jsonl',
    'data/nfcorpus/test_sft.jsonl',
]
for f in files:
    if os.path.exists(f):
        with open(f) as fh:
            n = sum(1 for _ in fh)
        size = os.path.getsize(f) // 1024
        print(f'✓ {f}: {n:,} records, {size} KB')
    else:
        print(f'✗ {f}: Missing!')

# Show first sample
with open('data/nfcorpus/train_sft.jsonl') as f:
    sample = json.loads(f.readline())
print(f'\n--- Sample ---')
print(f'Input length: {len(sample["input"])} chars')
print(f'Output: {sample["output"]}')
print(f'Relevance: {sample["relevance"]}')
print(f'Query ID: {sample["id"]}')
print(f'\nInput preview:\n{sample["input"][:500]}...')

✓ data/nfcorpus/train_sft.jsonl: 2,590 records, 11729 KB
✓ data/nfcorpus/val_sft.jsonl: 324 records, 1468 KB
✓ data/nfcorpus/test_sft.jsonl: 323 records, 1463 KB

--- Sample ---
Input length: 2266 chars
Output: 2, 4, 5, 1, 3
Relevance: [0, 1, 0, 1, 1]
Query ID: PLAIN-3

Input preview:
Below are 5 candidate passages for a medical information query. Rank them from most relevant to least relevant.
Output only the passage numbers separated by commas.

### Query:
Breast Cancer Cells Feed on Cholesterol

### Passages:
1. Systemic immunity-enhancing effects in healthy subjects following dietary consumption of the lactic acid bacterium Lactobacillus rhamnosus HN001.. OBJECTIVE: To determine the effects of the probiotic lactic acid bacterium, Lactobacillus rhamnosus HN001, on natural ...


---
## 2. LoRA Experiments

### E21: LoRA — NFCorpus × Qwen2.5-1.5B

In [ ]:
!source /etc/network_turbo
import os
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://huggingface.co'

from huggingface_hub import snapshot_download

print("Downloading Qwen...")
snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", cache_dir="/root/autodl-tmp/hf_cache", resume_download=True)
print("  Qwen done")

print("Downloading Llama...")
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", cache_dir="/root/autodl-tmp/hf_cache", resume_download=True)
print("  Llama done")

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证
下载 Qwen...


/root/miniconda3/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Qwen 完成
下载 Llama...


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Llama 完成


In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
os.makedirs('logs', exist_ok=True)
!{sys.executable} -u src/train/train.py --config configs/lora_nfcorpus_qwen.yaml 2>&1 | tee logs/lora_nfcorpus_qwen.log
print('Training complete')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/lora_nfcorpus_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/nfcorpus/train_sft.jsonl  (2,590 records)
Val     : /root/MLP/data/nfcorpus/val_sft.jsonl
Max input length : 1024
Max output length: 64
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 636.81it/s]
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Preparing datasets...
Tokenizing: 100%|███

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.makedirs('results/nfcorpus', exist_ok=True)
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_nfcorpus_qwen.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/lora_nfcorpus_qwen/predictions_test.jsonl --output results/nfcorpus/lora_qwen_test.json
print('Evaluation complete')
!cat results/nfcorpus/lora_qwen_test.json

Config     : configs/lora_nfcorpus_qwen.yaml
Task       : reranking
Split      : test  →  /root/MLP/data/nfcorpus/test_sft.jsonl
Output     : /root/MLP/outputs/lora_nfcorpus_qwen
Batch size : 16
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 574.84it/s]
Loading LoRA adapter: /root/MLP/outputs/lora_nfcorpus_qwen/final_adapter
Loaded 323 test records
The following generation flags are not valid an

### E22: LoRA — NFCorpus × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/lora_nfcorpus_llama.yaml 2>&1 | tee logs/lora_nfcorpus_llama.log
print('Training complete')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/lora_nfcorpus_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/nfcorpus/train_sft.jsonl  (2,590 records)
Val     : /root/MLP/data/nfcorpus/val_sft.jsonl
Max input length : 1024
Max output length: 64
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 368.74it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing datasets...
Tokenizing: 1

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_nfcorpus_llama.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/lora_nfcorpus_llama/predictions_test.jsonl --output results/nfcorpus/lora_llama_test.json
print('Evaluation complete')
!cat results/nfcorpus/lora_llama_test.json

Config     : configs/lora_nfcorpus_llama.yaml
Task       : reranking
Split      : test  →  /root/MLP/data/nfcorpus/test_sft.jsonl
Output     : /root/MLP/outputs/lora_nfcorpus_llama
Batch size : 16
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: meta-llama/Llama-3.2-1B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 374.82it/s]
Loading LoRA adapter: /root/MLP/outputs/lora_nfcorpus_llama/final_adapter
Loaded 323 test records
The following generation flags are not

---
## 3. Full Fine-Tuning Experiments

### E23: Full FT — NFCorpus × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_nfcorpus_qwen.yaml 2>&1 | tee logs/full_nfcorpus_qwen.log
print('Training complete')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_nfcorpus_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/nfcorpus/train_sft.jsonl  (2,590 records)
Val     : /root/MLP/data/nfcorpus/val_sft.jsonl
Max input length : 1024
Max output length: 64
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 650.59it/s]
Full fine-tuning: 1.54B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 324/324 [00:00<00:00, 518

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_nfcorpus_qwen.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/full_nfcorpus_qwen/predictions_test.jsonl --output results/nfcorpus/full_qwen_test.json
print('Evaluation complete')
!cat results/nfcorpus/full_qwen_test.json

Config     : configs/full_nfcorpus_qwen.yaml
Task       : reranking
Split      : test  →  /root/MLP/data/nfcorpus/test_sft.jsonl
Output     : /root/MLP/outputs/full_nfcorpus_qwen
Batch size : 16
Loading full fine-tuned model: /root/MLP/outputs/full_nfcorpus_qwen/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 498.10it/s]
Loaded 323 test records
The following generation flags are not valid and may be ignored: ['temperature', 'top_p

### E24: Full FT — NFCorpus × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_nfcorpus_llama.yaml 2>&1 | tee logs/full_nfcorpus_llama.log
print('Training complete')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_nfcorpus_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/nfcorpus/train_sft.jsonl  (2,590 records)
Val     : /root/MLP/data/nfcorpus/val_sft.jsonl
Max input length : 1024
Max output length: 64
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 384.52it/s]
Full fine-tuning: 1.24B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 324/324 [00:00<00:

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_nfcorpus_llama.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/full_nfcorpus_llama/predictions_test.jsonl --output results/nfcorpus/full_llama_test.json
print('Evaluation complete')
!cat results/nfcorpus/full_llama_test.json

Config     : configs/full_nfcorpus_llama.yaml
Task       : reranking
Split      : test  →  /root/MLP/data/nfcorpus/test_sft.jsonl
Output     : /root/MLP/outputs/full_nfcorpus_llama
Batch size : 16
Loading full fine-tuned model: /root/MLP/outputs/full_nfcorpus_llama/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 276.19it/s]
Loaded 323 test records
The following generation flags are not valid and may be ignored: ['temperature', 'to

---
## 4. Random Label Baseline

Shuffle the training set ranking outputs (labels) while keeping inputs unchanged, to verify whether the model genuinely learns query-passage relevance.

### 4.0 Generate Random Label Data

In [28]:
import json, random
from pathlib import Path

random.seed(42)

def shuffle_labels(input_path, output_path):
    records = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    outputs = [r['output'] for r in records]
    random.shuffle(outputs)
    for rec, new_out in zip(records, outputs):
        rec['output'] = new_out
        # Also update the 'text' field (full prompt with answer)
        rec['text'] = rec['input'] + new_out
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'  ✓ {output_path}: {len(records):,} records (labels shuffled)')

print('=== Generating Random Label NFCorpus Datasets ===')
shuffle_labels('data/nfcorpus/train_sft.jsonl', 'data/nfcorpus/train_sft_random.jsonl')
shuffle_labels('data/nfcorpus/val_sft.jsonl',   'data/nfcorpus/val_sft_random.jsonl')
print('\nDone. Test file is NOT shuffled (evaluate on real data).')

=== Generating Random Label NFCorpus Datasets ===
  ✓ data/nfcorpus/train_sft_random.jsonl: 2,590 records (labels shuffled)
  ✓ data/nfcorpus/val_sft_random.jsonl: 324 records (labels shuffled)

Done. Test file is NOT shuffled (evaluate on real data).


In [ ]:
# Verify mismatch rate
import json

def mismatch_rate(orig_path, random_path):
    with open(orig_path) as f1, open(random_path) as f2:
        orig = [json.loads(l)['output'] for l in f1 if l.strip()]
        rand = [json.loads(l)['output'] for l in f2 if l.strip()]
    return sum(a != b for a, b in zip(orig, rand)) / len(orig)

print('Mismatch rates (should be ~1.0):')
print(f'  NFCorpus train: {mismatch_rate("data/nfcorpus/train_sft.jsonl", "data/nfcorpus/train_sft_random.jsonl"):.4f}')
print(f'  NFCorpus val:   {mismatch_rate("data/nfcorpus/val_sft.jsonl", "data/nfcorpus/val_sft_random.jsonl"):.4f}')

Mismatch rates (should be ~1.0):
  NFCorpus train: 0.9205
  NFCorpus val:   0.9352


### E25: Random Label LoRA — NFCorpus × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_nfcorpus_qwen.yaml 2>&1 | tee logs/random_nfcorpus_qwen.log
print('Training complete')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_nfcorpus_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/nfcorpus/train_sft_random.jsonl  (2,590 records)
Val     : /root/MLP/data/nfcorpus/val_sft_random.jsonl
Max input length : 1024
Max output length: 64
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 691.48it/s]
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Preparing datasets...
Toke

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_nfcorpus_qwen.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/random_nfcorpus_qwen/predictions_test.jsonl --output results/nfcorpus/random_qwen_test.json
print('Evaluation complete')
!cat results/nfcorpus/random_qwen_test.json

Config     : configs/random_nfcorpus_qwen.yaml
Task       : reranking
Split      : test  →  /root/MLP/data/nfcorpus/test_sft.jsonl
Output     : /root/MLP/outputs/random_nfcorpus_qwen
Batch size : 16
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 645.27it/s]
Loading LoRA adapter: /root/MLP/outputs/random_nfcorpus_qwen/final_adapter
Loaded 323 test records
The following generation flags are not va

### E26: Random Label LoRA — NFCorpus × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_nfcorpus_llama.yaml 2>&1 | tee logs/random_nfcorpus_llama.log
print('Training complete')

/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/random_nfcorpus_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/nfcorpus/train_sft_random.jsonl  (2,590 records)
Val     : /root/MLP/data/nfcorpus/val_sft_random.jsonl
Max input length : 1024
Max output length: 64
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 402.33it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing datasets.

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_nfcorpus_llama.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/random_nfcorpus_llama/predictions_test.jsonl --output results/nfcorpus/random_llama_test.json
print('Evaluation complete')
!cat results/nfcorpus/random_llama_test.json

Config     : configs/random_nfcorpus_llama.yaml
Task       : reranking
Split      : test  →  /root/MLP/data/nfcorpus/test_sft.jsonl
Output     : /root/MLP/outputs/random_nfcorpus_llama
Batch size : 16
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: meta-llama/Llama-3.2-1B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 408.14it/s]
Loading LoRA adapter: /root/MLP/outputs/random_nfcorpus_llama/final_adapter
Loaded 323 test records
The following generation flags a

---
## 5. Summary of All NFCorpus Results

In [ ]:
import json, os

results = [
    ('LoRA  × Qwen',        'results/nfcorpus/lora_qwen_test.json'),
    ('LoRA  × Llama',       'results/nfcorpus/lora_llama_test.json'),
    ('Full FT × Qwen',     'results/nfcorpus/full_qwen_test.json'),
    ('Full FT × Llama',    'results/nfcorpus/full_llama_test.json'),
    ('Random × Qwen',      'results/nfcorpus/random_qwen_test.json'),
    ('Random × Llama',     'results/nfcorpus/random_llama_test.json'),
]

print(f'{"Experiment":<25} {"NDCG@5":>10} {"MAP@5":>10}')
print('-' * 50)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<25} {"Not completed":>10}')
        continue
    with open(path) as f:
        d = json.load(f)
    ndcg = d['ndcg@5']['mean'] if isinstance(d['ndcg@5'], dict) else d['ndcg@5']
    mapk = d['map@5']['mean'] if isinstance(d['map@5'], dict) else d['map@5']
    print(f'{name:<25} {ndcg:>10.4f} {mapk:>10.4f}')

实验                            NDCG@5      MAP@5
--------------------------------------------------
LoRA  × Qwen                  0.8943     0.8586
LoRA  × Llama                 0.8913     0.8509
Full FT × Qwen                0.8859     0.8451
Full FT × Llama               0.8820     0.8365
Random × Qwen                 0.7914     0.7012
Random × Llama                0.7873     0.6934


In [ ]:
import os, yaml

# ── Qwen baseline ──
os.makedirs('outputs/baseline_nfcorpus_qwen/final_model', exist_ok=True)
snap_qwen = '/root/autodl-tmp/hf_cache/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
for f in os.listdir(snap_qwen):
    src = os.path.join(snap_qwen, f)
    dst = os.path.join('outputs/baseline_nfcorpus_qwen/final_model', f)
    if not os.path.exists(dst):
        os.symlink(src, dst)

# ── Llama baseline ──
os.makedirs('outputs/baseline_nfcorpus_llama/final_model', exist_ok=True)
snap_llama = '/root/autodl-tmp/hf_cache/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6'
for f in os.listdir(snap_llama):
    src = os.path.join(snap_llama, f)
    dst = os.path.join('outputs/baseline_nfcorpus_llama/final_model', f)
    if not os.path.exists(dst):
        os.symlink(src, dst)

# ── Generate baseline configs (based on full config, remove lora section, change output dir) ──
for model_tag, model_name in [('qwen', 'Qwen/Qwen2.5-1.5B-Instruct'), ('llama', 'meta-llama/Llama-3.2-1B-Instruct')]:
    cfg = {
        'model': {'name': model_name, 'task': 'reranking'},
        'training': {'bf16': True, 'fp16': False, 'seed': 42},
        'data': {
            'train': f'data/nfcorpus/train_sft.jsonl',
            'val':   f'data/nfcorpus/val_sft.jsonl',
            'test':  f'data/nfcorpus/test_sft.jsonl',
            'max_input_length': 1024,
            'max_output_length': 64,
            'text_field': 'input',
            'label_field': 'output',
        },
        'output': {'dir': f'outputs/baseline_nfcorpus_{model_tag}'},
    }
    with open(f'configs/baseline_nfcorpus_{model_tag}.yaml', 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

print("Baseline setup complete")

Baseline 准备完成


In [ ]:
import sys, os
os.chdir('/root/MLP')

# Qwen baseline
!{sys.executable} -u src/evaluate/inference.py --config configs/baseline_nfcorpus_qwen.yaml --split test
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/baseline_nfcorpus_qwen/predictions_test.jsonl --output results/nfcorpus/baseline_qwen_test.json
print("Qwen baseline complete")
!cat results/nfcorpus/baseline_qwen_test.json

# Llama baseline
!{sys.executable} -u src/evaluate/inference.py --config configs/baseline_nfcorpus_llama.yaml --split test
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/baseline_nfcorpus_llama/predictions_test.jsonl --output results/nfcorpus/baseline_llama_test.json
print("Llama baseline complete")
!cat results/nfcorpus/baseline_llama_test.json

Config     : configs/baseline_nfcorpus_qwen.yaml
Task       : reranking
Split      : test  →  /root/MLP/data/nfcorpus/test_sft.jsonl
Output     : /root/MLP/outputs/baseline_nfcorpus_qwen
Batch size : 8
Loading full fine-tuned model: /root/MLP/outputs/baseline_nfcorpus_qwen/final_model
`torch_dtype` is deprecated! Use `dtype` instead!
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 623.66it/s]
Loaded 323 test records
The following generation flags are not valid and may be ignored: ['temperatu